# 남한 지도 픽셀아트 생성기 (레고용)

레고 픽셀아트용 저해상도 남한 지도를 생성합니다.

## 특징
- 구 경계 제거 → 시/도 경계만 표시
- 도/광역시 경계: 두꺼운 검정선
- 시/군 경계: 얇은 회색선
- 바다: 파란색 / 육지(남한·북한): 하얀색
- 작은 섬 자동 제거
- 해상도 지정 가능 (16 단위 권장: 96×128, 112×128, 112×144 등)

## 필요한 파일
- `skorea-municipalities-2018-geo.json` : 남한 시군구 경계 데이터
- `north_korea.geojson` : 북한 형태 데이터

## 필요한 패키지
```
pip install matplotlib shapely pillow
```

In [ ]:
# 1. 패키지 임포트 및 설정
import json
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from shapely.geometry import shape, box
from shapely.ops import unary_union
from PIL import Image

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

print("설정 완료")

In [ ]:
# 2. 설정값

# 데이터 파일 경로
DATA = r"skorea-municipalities-2018-geo.json"
NK   = r"north_korea.geojson"

# ── 출력 해상도 (픽셀) ──
# 16 단위로 설정 권장 (예: 96x128, 112x128, 112x144, 128x176)
# 지리 비율(세로/가로 ≒ 1.39)에 맞추려면:
#   가로 96  → 세로 128 (1.33)
#   가로 112 → 세로 144 (1.29) 또는 160 (1.43)
#   가로 128 → 세로 176 (1.38)
PX_W = 96    # 가로 픽셀 수
PX_H = 128   # 세로 픽셀 수

# 고해상도 렌더링 배수 (최종 이미지를 이 배수로 렌더링 후 다운샘플링)
# 클수록 정확하지만 느림 (기본 10)
RENDER_SCALE = 10

# ── 색상 ──
COLOR_SEA  = "#4a90d9"   # 바다 (파란색)
COLOR_LAND = "#ffffff"   # 육지 (남한·북한 하얀색)
COLOR_DO   = "#000000"   # 도/광역시 경계 (검정, 두껍게)
COLOR_SI   = "#888888"   # 시/군 경계 (회색, 얇게)

# ── 선 두께 (최종 픽셀 기준) ──
LW_DO_PX = 2.0   # 도/광역시 경계 (픽셀)
LW_SI_PX = 1.0   # 시/군 경계 (픽셀)

# ── 작은 섬 제거 ──
# 방식 1: ISLAND_MIN_AREA — 이 값(제곱도)보다 면적이 작은 섬은 제거
#   0.01 ≈ 100km², 0.02 ≈ 200km², 0.05 ≈ 500km²
ISLAND_MIN_AREA = 0.01

# 방식 2: KEEP_LARGEST_ONLY — 각 시/군에서 가장 큰 폴리곤(본토/본섬)만 남기고 나머지 섬 모두 제거
# True 면 해안선의 자잘한 섬이 완전히 사라짐 (제주도·거제도 등 큰 섬은 자동 보존)
KEEP_LARGEST_ONLY = True

# ── 북한 표시 여부 ──
# True: 북한 하단(개성 부근) 표시, False: 남한만 표시
# False로 설정하면 지도 범위가 남한에 맞춰져 태안반도가 왼쪽 끝이 됨
SHOW_NK = False

# 북한 절단 위도 (SHOW_NK=True일 때 이 위도 아래만 표시)
NK_CUT = 38.71740371170524

# 지리 비율 보정 (위도/경도 실제 거리 비)
LAT_LON_RATIO = 1.391 / 1.127

print(f"출력 해상도: {PX_W} x {PX_H} 픽셀")

In [ ]:
# 3. 데이터 로드 및 폴리곤 준비
with open(DATA, encoding="utf-8") as f:
    gj = json.load(f)

features = gj["features"]

# 광역시/특별시 코드 (구 경계 제거 → 시 단위로만)
METRO_CODES = {"11", "21", "22", "23", "24", "25", "26"}

# 시도별 폴리곤 (도/광역시 경계용)
sido_groups = {}
# 시/군별 폴리곤 (시/군 경계용)
city_groups = {}

for ft in features:
    name = ft["properties"]["name"]
    code = ft["properties"]["code"]
    if name == "울릉군":
        continue  # 울릉도/독도 제외
    if name == "옹진군":
        continue  # 백령도·인천 앞바다 섬 제외 (비율 개선)
    geom = shape(ft["geometry"])
    sido_groups.setdefault(code[:2], []).append(geom)

    if code[:2] in METRO_CODES:
        # 광역시: 구 없이 시 단위만
        city_groups.setdefault(name, []).append(geom)
    else:
        # 도: 시/군 단위로 병합 (구로 나뉜 시는 시 이름으로 병합)
        city = name[:name.index("시") + 1] if "시" in name else name
        city_groups.setdefault(city, []).append(geom)

# 병합 + 작은 섬 제거 헬퍼
def merge_and_filter(geoms, min_area, keep_largest=False):
    merged = unary_union(geoms)
    if merged.geom_type == "Polygon":
        polys = [merged]
    else:
        polys = list(merged.geoms)
    if keep_largest:
        # 가장 큰 폴리곤만 남기기
        if not polys:
            return None
        return max(polys, key=lambda p: p.area)
    filtered = [p for p in polys if p.area >= min_area]
    if not filtered:
        return None
    return unary_union(filtered)

# 시도 경계 (두껍게)
sido_boundaries = []
for code, geoms in sido_groups.items():
    merged = merge_and_filter(geoms, ISLAND_MIN_AREA, KEEP_LARGEST_ONLY)
    if merged:
        sido_boundaries.append(merged)

# 시/군 경계 (얇게)
city_boundaries = []
for city, geoms in city_groups.items():
    merged = merge_and_filter(geoms, ISLAND_MIN_AREA, KEEP_LARGEST_ONLY)
    if merged:
        city_boundaries.append(merged)

# 북한 (절단선 아래만) — SHOW_NK가 True일 때만 로드
nk_clip = None
if SHOW_NK:
    with open(NK, encoding="utf-8") as f:
        nk_gj = json.load(f)
    nk_geom = shape(nk_gj["geometry"])
    nk_clip = nk_geom.intersection(box(-180, -90, 180, NK_CUT))
    print(f"북한 폴리곤 로드 완료 (절단 위도 {NK_CUT})")
else:
    print("북한 미표시 (SHOW_NK=False)")

print(f"시도 경계: {len(sido_boundaries)}개")
print(f"시/군 경계: {len(city_boundaries)}개")

In [ ]:
# 4. 지도 그리기 + 다운샘플링 + 저장

# 전체 범위 계산 (남한 + 북한)
xs, ys = [], []
for g in sido_boundaries:
    minx, miny, maxx, maxy = g.bounds
    xs += [minx, maxx]
    ys += [miny, maxy]
if nk_clip is not None:
    nk_b = nk_clip.bounds
    xs += [nk_b[0], nk_b[2]]
    ys += [nk_b[1], nk_b[3]]

xmin, xmax = min(xs), max(xs)
ymin, ymax = min(ys), max(ys)

# 포항 동쪽 여백
xpad = (xmax - xmin) * 0.03
xmax += xpad

# 고해상도 렌더링 설정
render_w = PX_W * RENDER_SCALE
render_h = PX_H * RENDER_SCALE
dpi = 100
fig_w = render_w / dpi
fig_h = render_h / dpi

# 선 두께: 최종 픽셀 → matplotlib points 변환
# 1 point = dpi/72 픽셀, 최종 1픽셀 = RENDER_SCALE 렌더 픽셀
LW_DO = LW_DO_PX * RENDER_SCALE * 72 / dpi
LW_SI = LW_SI_PX * RENDER_SCALE * 72 / dpi

fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
ax.set_position([0, 0, 1, 1])  # 여백 없이 꽉 채우기
ax.set_aspect(LAT_LON_RATIO)

# 배경 (바다)
ax.set_facecolor(COLOR_SEA)
fig.set_facecolor(COLOR_SEA)

# 북한 육지 (하얀색) — SHOW_NK가 True일 때만
if nk_clip is not None:
    for poly in ([nk_clip] if nk_clip.geom_type == "Polygon" else list(nk_clip.geoms)):
        x, y = poly.exterior.xy
        ax.fill(x, y, facecolor=COLOR_LAND, edgecolor="none", zorder=1)

# 남한 육지 (하얀색)
for g in sido_boundaries:
    for poly in ([g] if g.geom_type == "Polygon" else list(g.geoms)):
        x, y = poly.exterior.xy
        ax.fill(x, y, facecolor=COLOR_LAND, edgecolor="none", zorder=2)

# 시/군 경계 (얇은 회색선)
for g in city_boundaries:
    for poly in ([g] if g.geom_type == "Polygon" else list(g.geoms)):
        x, y = poly.exterior.xy
        ax.plot(x, y, color=COLOR_SI, linewidth=LW_SI, zorder=3)

# 시도/광역시 경계 (두꺼운 검정선)
for g in sido_boundaries:
    for poly in ([g] if g.geom_type == "Polygon" else list(g.geoms)):
        x, y = poly.exterior.xy
        ax.plot(x, y, color=COLOR_DO, linewidth=LW_DO, zorder=4)

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.axis("off")

# 고해상도로 렌더링
temp_path = "_pixelart_hires.png"
plt.savefig(temp_path, dpi=dpi, facecolor=fig.get_facecolor())
plt.close()

# NEAREST 다운샘플링 → 픽셀아트
img = Image.open(temp_path)
img = img.resize((PX_W, PX_H), Image.NEAREST)

OUT = f"pixelart_{PX_W}x{PX_H}.png"
img.save(OUT)
os.remove(temp_path)

print(f"saved: {OUT}")
print(f"size: {img.size[0]} x {img.size[1]} px")

## 결과 확인

생성된 `pixelart_{가로}x{세로}.png` 파일을 확인하세요.

## 설정 변경 (2번 셀)

| 항목 | 변수 | 설명 |
|------|------|------|
| 해상도 | `PX_W`, `PX_H` | 출력 픽셀 크기 (16 단위 권장) |
| 렌더 배수 | `RENDER_SCALE` | 클수록 정확하지만 느림 (기본 10) |
| 바다 색 | `COLOR_SEA` | 기본 파란색 |
| 육지 색 | `COLOR_LAND` | 기본 하얀색 |
| 도 경계 색/두께 | `COLOR_DO`, `LW_DO_PX` | 검정, 2픽셀 |
| 시 경계 색/두께 | `COLOR_SI`, `LW_SI_PX` | 회색, 1픽셀 |
| 섬 제거 | `ISLAND_MIN_AREA` | 이 값보다 작은 섬은 제거 |

## 해상도 예시 (16 단위)

| 가로 | 세로 | 비율(세로/가로) |
|------|------|------|
| 96 | 128 | 1.33 |
| 112 | 144 | 1.29 |
| 112 | 160 | 1.43 |
| 128 | 176 | 1.38 |
| 144 | 192 | 1.33 |

지리 비율(세로/가로 ≒ 1.39)에 가장 가까운 조합을 선택하면 여백이 최소화됩니다.